# J2 : vérification des calculs

Le guide demande que « les valeurs soient vérifiées » et que les règles de qualité soient explicites. Ce notebook montre **comment** on vérifie
(tolérance, comparaison par clé) et **ce que couvrent les tests**, en les exécutant réellement.

In [1]:
from pathlib import Path
import json, sys, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Racine du projet (ce notebook est dans notebooks/) : on importe le code de src/, on ne le recopie pas.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
TMP = Path(tempfile.mkdtemp(prefix="bigdata_nb_"))   # fichiers temporaires : data/ n'est jamais modifié

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "figure.figsize": (9, 3.6)})
print("Racine du projet :", ROOT)

import subprocess, xml.etree.ElementTree as ET
from importlib import metadata
from src.common import ABS_TOL, REL_TOL, compare_results, within_tolerance

Racine du projet : /home/claude/BigData_Project


## 1. La règle de tolérance (fixée avant toute comparaison)

$$|v-r|\le \text{atol}+\text{rtol}\times|r|\qquad\text{avec } \text{atol}=\text{rtol}=10^{-9}$$

$v$ est la valeur à contrôler, $r$ la référence Python. Règle explicite pour les valeurs manquantes : deux NaN sont considérés égaux, un NaN face à un nombre ne l'est jamais.

In [2]:
cas = [(1000 + 0.9e-6, 1000.0), (1000 + 1.1e-6, 1000.0), (0.0, 5e-10), (0.0, 2e-9), (np.nan, np.nan), (np.nan, 1.0)]
tab = pd.DataFrame([{"v": v, "r (référence)": r, "limite": ABS_TOL + REL_TOL * abs(r) if r == r else np.nan,
                     "|v - r|": abs(v - r) if v == v and r == r else np.nan,
                     "dans la tolérance": bool(within_tolerance(v, r))} for v, r in cas])
display(tab)

,v,r (référence),limite,|v - r|,dans la tolérance
0,1000.000001,1.000000e+03,1.001000e-06,9.000000e-07,True
1,1000.000001,1.000000e+03,1.001000e-06,1.100000e-06,False
2,0.000000,5.000000e-10,1.000000e-09,5.000000e-10,True
3,0.000000,2.000000e-09,1.000000e-09,2.000000e-09,False
4,NaN,NaN,NaN,NaN,True
5,NaN,1.000000e+00,2.000000e-09,NaN,False


## 2. La comparaison par clé

On compare **après tri par clé** : mêmes clés, même nombre de lignes, valeurs proches. Comparer seulement le nombre de lignes ne suffirait pas.

In [3]:
ref = pd.DataFrame({"fenetre": [1, 2, 3], "moyenne": [10.0, 20.0, 30.0]})

melange = ref.sample(frac=1.0, random_state=0).reset_index(drop=True)          # mêmes données, autre ordre
compare_results(melange, ref, ["fenetre"])
print("1) Même contenu dans un autre ordre : accepté (tri par clé).")

for nom, faux in {"valeur hors tolérance": ref.assign(moyenne=[10.0, 20.0, 30.1]),
                  "clé différente": ref.assign(fenetre=[1, 2, 4]),
                  "ligne manquante": ref.iloc[:2]}.items():
    try:
        compare_results(faux, ref, ["fenetre"])
        print(f"   {nom} : NON détecté (anormal)")
    except AssertionError as e:
        print(f"2) {nom} : refusé -> {str(e)[:70]}")

1) Même contenu dans un autre ordre : accepté (tri par clé).
2) valeur hors tolérance : refusé -> colonne moyenne hors tolérance sur 1 ligne(s)
2) clé différente : refusé -> clés.iloc[:, 0] (column name="fenetre") are different

clés.iloc[:, 0]
2) ligne manquante : refusé -> nombre de lignes différent : 2 vs 3


## 3. Exécution réelle des tests

Les tests utilisent des données synthétiques construites pour provoquer chaque cas, et comparent chaque résultat à un calcul indépendant (pandas ou NumPy).
Ils sont lancés ici avec `pytest`, et le résultat est lu dans le fichier JUnit.

In [4]:
junit = TMP / "junit.xml"
proc = subprocess.run([sys.executable, "-m", "pytest", "-o", "addopts=", "-q", "-p", "no:cacheprovider", f"--junitxml={junit}"],
                      cwd=ROOT, capture_output=True, text=True)
print(proc.stdout.strip().splitlines()[-1])

rows = []
for tc in ET.parse(junit).getroot().iter("testcase"):
    echec = tc.find("failure") is not None or tc.find("error") is not None
    rows.append({"fichier": tc.get("classname").split(".")[-1], "test": tc.get("name"), "ok": not echec})
tests = pd.DataFrame(rows)
tests["nom"] = tests.test.str.replace(r"\[.*\]", "", regex=True)
print(f"{len(tests)} cas de test exécutés, {int(tests.ok.sum())} réussis")
display(tests.groupby("fichier").agg(cas=("ok", "size"), reussis=("ok", "sum")))

71 passed in 26.28s
71 cas de test exécutés, 71 réussis


,cas,reussis
fichier,,
test_common,10,10
test_datamining,15,15
test_metropt3,23,23
test_ukdale,23,23


### Couverture par rapport aux cas demandés par le guide (§9)

Un test peut compter dans plusieurs catégories.

In [5]:
categories = {
    "Horodatage exactement sur une borne": ["boundary", "ending_exactly"],
    "Fenêtre coupée entre deux blocs": ["cut_between", "window_cut"],
    "Dernière fenêtre incomplète": ["incomplete"],
    "Partitions (blocs) de tailles inégales": ["chunks_of_different", "random_block_sizes", "single_sample_blocks", "flac_read_by_blocks"],
    "Périodes sans observations": ["without_observations", "missing_window", "gap"],
    "Doublons": ["duplicate"],
    "Valeurs invalides et valeurs exclues consignées": ["invalid", "excluded", "unreadable", "nan_indicator"],
    "Cas à une observation": ["single_observation"],
    "Exemples numériques du guide": ["guide_p1"],
    "Écart-type de population et stabilité numérique": ["std", "constant_window", "stable", "merge_moments"],
    "Tolérance et comparaison par clé": ["tolerance", "compare_results"],
    "Calibration et association des canaux": ["calibration", "channels", "implausible", "known_signal", "wrong_sample"],
    "Traitement B : durée et non lignes, pas de fuite du futur": ["history", "no_future", "variation", "order_of_the_input"],
    "Fichiers d'origine jamais modifiés": ["never_modified"],
}
res = []
for cat, mots in categories.items():
    sel = tests[tests.nom.apply(lambda s: any(m in s for m in mots))]
    res.append({"Cas": cat, "Tests": len(sel), "Réussis": int(sel.ok.sum())})
display(pd.DataFrame(res))
print("Non couvert : fenêtre coupée entre deux FICHIERS (UK-DALE : un seul fichier lu à la fois).")

,Cas,Tests,Réussis
0,Horodatage exactement sur une borne,2,2
1,Fenêtre coupée entre deux blocs,2,2
2,Dernière fenêtre incomplète,3,3
3,Partitions (blocs) de tailles inégales,17,17
4,Périodes sans observations,3,3
5,Doublons,2,2
6,Valeurs invalides et valeurs exclues consignées,5,5
7,Cas à une observation,2,2
8,Exemples numériques du guide,2,2
9,Écart-type de population et stabilité numérique,8,8


Non couvert : fenêtre coupée entre deux FICHIERS (UK-DALE : un seul fichier lu à la fois).


## 4. Reproductibilité : les versions utilisées

Le guide demande de figer les versions effectivement validées (`requirements.txt`).

In [6]:
req = {}
for ligne in (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    if "==" in ligne and not ligne.startswith("#"):
        nom, ver = ligne.split("==")
        req[nom.strip()] = ver.strip()
tab = pd.DataFrame([{"paquet": p, "figé (requirements.txt)": v, "installé": metadata.version(p),
                     "identique": metadata.version(p) == v} for p, v in req.items()])
display(tab)
print("Python :", sys.version.split()[0], "(référence J1 : 3.14)")

,paquet,figé (requirements.txt),installé,identique
0,pandas,2.3.3,2.3.3,True
1,numpy,2.3.5,2.3.5,True
2,soundfile,0.14.0,0.14.0,True
3,pytest,9.2.0,9.2.0,True


Python : 3.13.18 (référence J1 : 3.14)


## 5. Ce que ces vérifications prouvent, et ce qu'elles ne prouvent pas

- Elles prouvent que **le code** applique les formules, les règles de fenêtrage et la tolérance, y compris dans les cas limites.
- Elles ne prouvent **pas** que les vrais fichiers donnent les valeurs attendues : cette vérification se fait en lançant les scripts sur les données réelles
  et en comparant les rapports de qualité à la fiche d'audit J1 (voir les notebooks 01 et 02, sections « Rapport de qualité » et « Contrôles de qualité »).
- Elles ne remplacent pas la comparaison Python / PySpark du jalon J3.